# Pipeline walkthrough

Loads a sample day from `data/samples/` (GeoJSON or GPX) and runs the real `app` code one step at a time,
printing what each step produced. The order is the same as `run_pipeline` in `app/pipeline/run.py`, followed by
what `app/services/ingest_service.py` does with the result.

To try other thresholds, edit `app/config/pipeline.py` or `app/config/io_types.py`, restart the kernel and run all cells.

Needs the road centrelines (`data/road_centrelines/RoadCentreLine.geojson`, see the README). expo_calx is only
called if `CALL_EXPO_CALX` is set to `True` below (needs `.env`); the notebook stores its rows in its own
`walkthrough.db` (recreated on every run), never in the real `exposure.db`.

In [ ]:
import os
import sys
from collections import Counter
from pathlib import Path

# Separate DB file so this notebook never touches the real exposure.db.
# Must be set before anything reads app settings.
WALKTHROUGH_DB = Path("walkthrough.db")
os.environ["EXPOSURE_DB_PATH"] = str(WALKTHROUGH_DB)

# This notebook lives in notebooks/, one level below the repo root.
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.clients import praise_client
from app.db import exposure_repository
from app.db.connection import init_db
from app.pipeline.classify import classify_runs
from app.pipeline.models import Track
from app.pipeline.motion import interval_motions
from app.pipeline.mtr import find_mtr
from app.pipeline.parsers import parse_pings
from app.pipeline.parsers.gpx import read_gpx
from app.pipeline.points import build_points
from app.pipeline.request_rows import to_request_rows
from app.pipeline.runs import build_runs, to_runs
from app.pipeline.spikes import remove_spikes
from app.pipeline.time_split import split_by_hour, to_hk
from app.services.ingest_service import build_db_rows
from app.services.road_service import get_road_index


def hk(utc):
    return to_hk(utc).strftime("%m-%d %H:%M:%S")


def print_table(rows, limit=60):
    """Print a list of dicts as aligned columns (first `limit` rows)."""
    if not rows:
        print("(none)")
        return
    columns = list(rows[0])
    cells = [[str(r[c]) for c in columns] for r in rows[:limit]]
    widths = [max(len(c), *(len(row[i]) for row in cells)) for i, c in enumerate(columns)]
    print("  ".join(c.ljust(w) for c, w in zip(columns, widths)))
    for row in cells:
        print("  ".join(v.ljust(w) for v, w in zip(row, widths)))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more")

## Load a sample day and the inputs

Change `DATA_PATH` to try another day (`.geojson` or `.gpx`). `PLACE_LABELS` is empty here; put
`PlaceLabel(...)` objects in it (`app/pipeline/place_labels.py`) to try labels.

In [ ]:
import json

DATA_PATH = REPO_ROOT / "data" / "samples" / "20260901.geojson"
PID = DATA_PATH.stem

if DATA_PATH.suffix == ".gpx":
    records = read_gpx(DATA_PATH.read_bytes())
else:
    records = json.loads(DATA_PATH.read_text(encoding="utf-8"))["features"]
print(f"{len(records)} records loaded from {DATA_PATH.name}")

roads = get_road_index()
PLACE_LABELS = []
print(f"{len(roads)} road centrelines, {len(PLACE_LABELS)} place labels")

## Input: `parse_pings` - only time, lng and lat are read

In [ ]:
pings = parse_pings(records)
print(f"{len(pings)} pings, {hk(pings[0].utc)} -> {hk(pings[-1].utc)} HK")
print_table([{"time_hk": hk(p.utc), "lng": p.lng, "lat": p.lat} for p in pings[:5]])

## Step 0 - `remove_spikes`: drop GPS points that jump away and straight back

In [ ]:
track = Track.from_pings(remove_spikes(pings))
print(f"{len(track)} pings remain ({len(pings) - len(track)} spikes dropped)")

## Step 1 - `interval_motions`: jump / still / walk / vehicle per interval

Interval *i* is the time from ping *i-1* to ping *i*.

In [ ]:
motions = interval_motions(track)
minutes = Counter()
for motion, dt in zip(motions, track.dt_s):
    minutes[motion.value] += dt / 60
print({m: round(v, 1) for m, v in minutes.most_common()})
print(f"{len(to_runs(motions))} raw runs before clean-up")

## Step 2 - `build_runs`: runs of the same motion, cleaned up (stops, blips, same-place stays, GPS loss)

In [ ]:
def run_rows(runs):
    return [
        {"motion": r.motion.value, "start_hk": hk(track.pings[r.first_ping].utc), "end_hk": hk(track.pings[r.last].utc),
         "minutes": round(r.seconds(track) / 60, 1), "pings": r.last - r.first_ping + 1}
        for r in runs
    ]


runs = build_runs(track, motions)
print(f"{len(runs)} runs")
print_table(run_rows(runs))

## Step 3 - `find_mtr`: rail test (distance to road), MTR time extension, interchanges

`road_med_m` is the median distance of the run's pings to the nearest road centreline; a vehicle run inside
Hong Kong with 15 m or more becomes `rail`.

In [ ]:
from statistics import median

nearest_roads = roads.nearest(track.pings)
runs = find_mtr(track, runs, nearest_roads)
print_table([
    {**row, "road_med_m": round(median(n.distance_m for n in nearest_roads[r.first_ping:r.last + 1]), 1)}
    for r, row in zip(runs, run_rows(runs)) if r.motion.value in ("vehicle", "rail", "jump", "interchange")
])

## Step 4 - `classify_runs`: the decision tree (place labels first)

In [ ]:
classified = classify_runs(track, runs, PLACE_LABELS)
print_table([
    {"io": c.io, "reason": c.reason.value, **row} for c, row in zip(classified, run_rows([c.run for c in classified]))
])

## Step 5 - `build_points` + `split_by_hour`

A stay becomes one point; a route keeps its Douglas-Peucker-thinned pings with heading and (Vehicle) road name.
Each point's time is cut at HK clock hours.

In [ ]:
points = build_points(track, classified, nearest_roads)
slices = split_by_hour(points)
print(f"{len(track)} pings -> {len(points)} points -> {len(slices)} hourly slices")
print("points per io:", dict(Counter(p.io for p in points)))
print_table([
    {"start_hk": hk(p.start_utc), "io": p.io, "kind": p.kind.value, "reason": p.reason.value,
     "lng": round(p.lng, 5), "lat": round(p.lat, 5), "compass": p.compass, "road": p.road_name}
    for p in points
], limit=40)

In [ ]:
minutes = Counter()
for s in slices:
    minutes[s.point.io] += s.delta_t_hours * 60
print("minutes per io:", {io: round(m, 1) for io, m in minutes.most_common()})
span_min = (pings[-1].utc - pings[0].utc).total_seconds() / 60
print(f"total {sum(minutes.values()):.1f} min = first-to-last ping {span_min:.1f} min")

## expo_calx request rows (inside Hong Kong only)

In [ ]:
hk_slices = [s for s in slices if s.point.in_hk]
rows = to_request_rows(PID, hk_slices)
print(f"{len(rows)} rows -> {praise_client.count_calls(len(rows))} expo_calx call(s)")
for row in rows[:5]:
    print(row)

## Call expo_calx (optional) and store in the notebook's own DB

In [ ]:
CALL_EXPO_CALX = False  # True makes real calls (needs .env; each call takes about 20-30 s)

if CALL_EXPO_CALX:
    results = praise_client.calculate_exposure(rows)
else:
    results = [praise_client.ExposureResult(t=r[0], pid=r[1], value=None, raw=None) for r in rows]
print(f"{len(results)} results")

WALKTHROUGH_DB.unlink(missing_ok=True)  # start from an empty DB with the current schema
init_db()
exposure_repository.upsert_hourly(build_db_rows(PID, slices, results))

## What `GET /exposure/hourly/{pid}` returns

In [ ]:
print_table(exposure_repository.get_hourly(PID), limit=20)